# PyCaret — MLOps
### Part 6

Beyond modeling, PyCaret has an **MLOps** story: experiment tracking with **MLflow**, model
packaging, deployment artifacts, and a path to serving via an API. This notebook walks the MLOps
lifecycle around a PyCaret classification model.

**Covered:**
1. `setup(log_experiment=True)` — automatic **MLflow** experiment logging
2. Training with tracked runs
3. Inspecting the MLflow run data
4. `save_model` / `load_model` — portable pipeline artifacts
5. `create_api` / `create_docker` — productionization scaffolding
6. `predict_model` on new data (batch inference)

> Runtime: CPU is enough. `Runtime → Run all`.

In [ ]:
!pip install -q pycaret mlflow
print("Installed. Restart runtime if prompted, then Run all.")

In [ ]:
import pycaret, mlflow, pandas as pd
print("PyCaret:", pycaret.__version__, "| MLflow:", mlflow.__version__)

## 1. Setup with experiment tracking

`log_experiment=True` + an `experiment_name` makes PyCaret log every trained model, its params,
metrics, and artifacts to MLflow automatically.

In [ ]:
from pycaret.datasets import get_data
from pycaret.classification import (setup, compare_models, tune_model,
                                    finalize_model, predict_model,
                                    save_model, load_model, create_api)

data = get_data("diabetes")

s = setup(
    data=data,
    target="Class variable",
    session_id=42,
    log_experiment=True,
    experiment_name="diabetes_mlops",
    verbose=False,
)
print("Setup complete with MLflow logging enabled.")

## 2. Train models — each run is logged to MLflow

In [ ]:
best = compare_models(n_select=1)
tuned = tune_model(best)

## 3. Inspect MLflow runs

Everything PyCaret trained is now queryable through the MLflow tracking API.

In [ ]:
runs = mlflow.search_runs(experiment_names=["diabetes_mlops"])
print("Logged runs:", len(runs))
cols = [c for c in runs.columns
        if c.startswith("metrics.") or c in ("run_id", "status")]
runs[cols].head(10)

In [ ]:
# Launch the MLflow UI (optional in Colab). Shows runs, metrics, artifacts.
# In Colab you can tunnel it; here we just show how:
print("To view the MLflow UI locally: !mlflow ui --port 5000")
print("In Colab, use pyngrok or Colab's port forwarding to expose port 5000.")

## 4. Finalize & persist the pipeline

`save_model` writes the **entire pipeline** (preprocessing + model) to a `.pkl` — the deployable
artifact. `load_model` restores it anywhere with no retraining.

In [ ]:
final = finalize_model(tuned)
save_model(final, "diabetes_mlops_pipeline")

reloaded = load_model("diabetes_mlops_pipeline")
print("Reloaded pipeline type:", type(reloaded))

## 5. Batch inference on new data

Simulate incoming production data and score it with the reloaded pipeline.

In [ ]:
new_data = data.drop(columns=["Class variable"]).sample(5, random_state=7)
scored = predict_model(reloaded, data=new_data)
scored

## 6. Productionization scaffolding

PyCaret can generate deployment scaffolding from a finalized model:

```python
# FastAPI service exposing /predict
create_api(final, "diabetes_api")
# -> writes diabetes_api.py ; run with: !python diabetes_api.py

# Dockerfile + requirements to containerize that API
create_docker("diabetes_api")
# -> writes Dockerfile and requirements.txt
```

Together these take you from a trained pipeline to a containerized REST endpoint. (We show the
calls here; actually serving requires running the generated server, which is awkward inside a
notebook — do it in a terminal / container.)

In [ ]:
# Generate the FastAPI scaffold (writes diabetes_api.py)
try:
    create_api(final, "diabetes_api")
    print("Wrote diabetes_api.py")
    import os
    print("Files:", [f for f in os.listdir('.') if f.endswith(('.py', '.pkl'))])
except Exception as e:
    print("create_api needs 'fastapi'/'uvicorn' installed:", e)

## Summary

The PyCaret MLOps lifecycle:
1. **Track** — `log_experiment=True` logs every run to MLflow (params, metrics, artifacts).
2. **Compare** — query runs via `mlflow.search_runs` to pick the best.
3. **Package** — `save_model`/`load_model` persist the full pipeline as a portable artifact.
4. **Serve** — `create_api` + `create_docker` scaffold a containerized REST service.
5. **Infer** — `predict_model` scores new/batch data with the saved pipeline.

This closes the loop from experimentation to a deployable, trackable, reproducible model.